 # Day 6 (Thu Aug 20) — Let's build the GPT Tokenizer (2h13m)

He types everything live, so this is a scratchpad, not a starter. His filled colab + minbpe repo are answer keys — stay out of them until it's written.

**why this day matters most:** 28 of the 48 CS336 tests are tokenizer tests (`test_train_bpe.py` 3 + `test_tokenizer.py` 25). It also unblocks the TinyStories run — nothing trains until my tokenizer can encode the corpus.

**arc (video chapters):**
- 14:56 unicode code points · 18:15 UTF-8 and why bytes
- 23:50 BPE algorithm · 27:02 implementation starts
- 28:35 get_stats · 30:36 merge · 34:58 the training loop + compression ratio
- 42:47 decode · 48:21 encode
- 57:36 **regex splitting** — the part that makes it a real tokenizer
- 1:11:38 tiktoken, GPT-2 vs GPT-4 patterns · 1:18:26 special tokens
- 1:25:28 exercise time (build your own GPT-4 tokenizer)
- 1:51:41 the quirks: why models can't spell, 9.11 vs 9.9, SolidGoldMagikarp

**targets:**
1. `train(text, vocab_size)` → vocab + merges. CS336 wants 500-token vocab on `corpus.en` in **under 1.5 seconds** — naive O(n²) passes correctness and fails the clock
2. `encode` / `decode` round-trip on unicode, and matching tiktoken exactly
3. special tokens that never get merged into
4. consolidate into `bpe.py`, wire into `05-cs336/assignment1-basics/tests/adapters.py`

**rule:** watch a chapter → close it → write it here → only then compare to the answer key.

**corpora:** `data/taylorswift.txt` (186KB, karpathy's default for the exercise) · `data/unicode_torture.txt` (fullwidth, circled letters, flag emoji — the round-trip has to survive it). Bigger: `05-cs336/.../tests/fixtures/corpus.en` for the 1.5s speed test.

In [44]:
# hyperparams
VOCAB_SIZE = 276 # the desired final vocabulary size

In [45]:
from pathlib import Path

D = Path("data")
text = (D / "taylorswift.txt").read_text()          # training corpus
torture = (D / "unicode_torture.txt").read_text()   # round-trip must survive this
tokens = text.encode("utf-8")

print(len(text), "chars |", len(tokens), "utf-8 bytes")
print(torture)

185561 chars | 185768 utf-8 bytes
Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺🇳🇮🇨🇴🇩🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide.


In [46]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
  newids = []
  i = 0
  while i < len(ids):
    if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
      newids.append(idx)
      i += 2
    else:
      newids.append(ids[i])
      i += 1
  return newids

In [47]:
num_merges = VOCAB_SIZE - 256 # 256 since there are 256 initial vocab size
ids = list(tokens) # copy so we don't destroy the original list

merges = {} # (int, int) -> int
for i in range(num_merges):
  stats = get_stats(ids)
  pair = max(stats, key=stats.get)
  idx = 256 + i
  print(f"merging {pair} into a new token {idx}")
  ids = merge(ids, pair, idx)
  merges[pair] = idx

merging (101, 32) into a new token 256
merging (44, 32) into a new token 257
merging (100, 32) into a new token 258
merging (46, 32) into a new token 259
merging (114, 32) into a new token 260
merging (50, 48) into a new token 261
merging (115, 32) into a new token 262
merging (105, 110) into a new token 263
merging (111, 110) into a new token 264
merging (114, 105) into a new token 265
merging (116, 32) into a new token 266
merging (116, 104) into a new token 267
merging (101, 258) into a new token 268
merging (257, 261) into a new token 269
merging (97, 110) into a new token 270
merging (97, 114) into a new token 271
merging (101, 260) into a new token 272
merging (121, 32) into a new token 273
merging (97, 108) into a new token 274
merging (267, 256) into a new token 275


In [49]:
print(merge([5, 6, 6, 7, 9, 1], (6, 7), 99))

[5, 6, 99, 9, 1]


In [50]:
print("tokens length:", len(tokens))
print("ids length:", len(ids))
print(f"compression ratio: {len(tokens) / len(ids):.2f}X")

tokens length: 185768
ids length: 147440
compression ratio: 1.26X


In [48]:
def decode(ids):
    """Given a list of ids, what is the string"""
    

In [51]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # GPT-4 tokenizer
print(enc.encode("안녕하세요 👋 (hello in Korean!)"))
print(enc.decode(enc.encode("안녕하세요 👋 (hello in Korean!)")) == "안녕하세요 👋 (hello in Korean!)")
# match the above for your own tokenizer, and also implement a train() function

ModuleNotFoundError: No module named 'tiktoken'